# Trial Batch Runner — Per-Model Subtrials with Risk-Aware Speed
Run every saved `TrialDefinition` **twice**: once with the **E2E** model and once with the **Composed BDU-GRU** model. Each run does **streaming inference** (the predictor consumes each captured RGB frame as the simulation ticks) and feeds the latest predicted risk back into a per-model speed function:

> **v' = f(v, r)** — `v` is the base target speed (km/h), `r` is the model's latest predicted risk (or `None` during burn-in / on prediction errors).

You configure one lambda per model below. The runner calls it every tick; the modifier reads whatever was the most recent prediction and returns the new target speed.

Each subtrial:
- saves ground truth + that model's `predicted_risk` (renamed to `predicted_risk_<model>`) into `dataset.jsonl`,
- lands at `MIREIA/trials/<trial>/runs/<timestamp>_e2e/` or `..._composed/`.

`SAVE_FRAMES=True` keeps dashcam (RGB) **and** top-down frames on disk (just like `trial_demo.ipynb`); `False` keeps only the JSONL and removes the run's `images/` after each subtrial.

## 0 — CARLA Launch
Skip if CARLA is already running.

In [ ]:
import subprocess

AUTO_LAUNCH_CARLA = True
CARLA_EXE = "CarlaUE4.exe"

if AUTO_LAUNCH_CARLA:
    subprocess.Popen(CARLA_EXE, shell=True)
    print(f"Launched {CARLA_EXE}")
else:
    print("AUTO_LAUNCH_CARLA is False. Assuming CARLA is already running.")

## 1 — Imports

In [ ]:
import json
import os
import shutil
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import torch
from ultralytics import YOLO

from MIREIA.config import Config
from MIREIA.data_collection.dataset_utils import load_jsonl_records
from MIREIA.perception import (
    DepthAnythingV2Estimator,
    FeatureIntegrator,
    QueuedComposedBDUGRURiskInference,
    QueuedE2ERiskInference,
    create_environment_classifier_predictor,
    load_road_segmentation_model,
)
from MIREIA.simulation.trials import EgoTrialConfig, TrialDefinition, TrialRunner

## 2 — Discover Trials
Lists every `trial.json` under `PATH_TO_TRIALS`. Use `INCLUDE_PREFIXES` / `EXCLUDE_TRIALS` to limit the run.

In [ ]:
trials_root = Path(Config.PATH_TO_TRIALS)
all_trials = sorted(p.parent.name for p in trials_root.glob("*/trial.json") if p.is_file())

# Filtering (leave INCLUDE_PREFIXES empty to include every trial)
INCLUDE_PREFIXES: list[str] = []   # e.g. ["auto_"] to only run auto-generated trials
EXCLUDE_TRIALS:   list[str] = []   # exact trial names to skip

if INCLUDE_PREFIXES:
    selected_trial_names = [n for n in all_trials if any(n.startswith(p) for p in INCLUDE_PREFIXES)]
else:
    selected_trial_names = list(all_trials)
selected_trial_names = [n for n in selected_trial_names if n not in EXCLUDE_TRIALS]

print(f"Found {len(all_trials)} trial(s); {len(selected_trial_names)} selected to run:")
for n in selected_trial_names:
    print(f"  - {n}")

## 3 — Per-Model Speed Functions and Subtrial Configs
Each model gets its own `v' = f(v, r)` lambda. They receive:

- `v`: the base target speed in km/h (constant per ego config — `target_speed_kmh`).
- `r`: the latest predicted risk from the model. `None` while the temporal queue is still warming up (burn-in) or on a transient predictor error.

Edit the lambdas below to change how each model reacts to its own risk estimate. Examples:
- Constant slow-down: `lambda v, r: 0.75 * v`
- Brake harder when risk is high: `lambda v, r: v * (1.0 - 0.7 * r) if r is not None else v`
- Floor at 25 % of the base speed: `lambda v, r: max(0.25 * v, v * (1.0 - 0.5 * r)) if r is not None else v`

`SAVE_FRAMES = True` keeps dashcam + top-down frames on disk; `False` keeps only JSONL (RGB is still written during the run because the streaming predictor needs it, then `images/` is wiped).

In [ ]:
# --- v' = f(v, r) lambdas, one per model ------------------------------------
def e2e_speed_fn(v: float, r: float | None) -> float:
    if r is None:
        return v
    return max(0.25 * v, v * (1.0 - 0.5 * r))

def composed_speed_fn(v: float, r: float | None) -> float:
    if r is None:
        return v
    return max(0.25 * v, v * (1.0 - 0.5 * r))

# --- Frame retention --------------------------------------------------------
SAVE_FRAMES              = False
TOPDOWN_RESOLUTION       = (1024, 1024)
TOPDOWN_FOV              = 95.0
TOPDOWN_ALIGN_RISK       = True
TOPDOWN_CAPTURE_DELAY    = 2

MAX_STEPS_PER_TRIAL = 6000
TARGET_SPEED_KMH    = 20.0   # base target speed used by both ego configs

e2e_ego_cfg = EgoTrialConfig(
    name="e2e",
    ego_blueprint="vehicle.lincoln.mkz_2020",
    target_speed_kmh=TARGET_SPEED_KMH,
    speed_multiplier=1.0,                # left at 1.0 -- the lambda does the modulation
    use_vehicle_camera_defaults=True,
    controller_mode="behavior_agent",
    controller_behavior="normal",
)

composed_ego_cfg = EgoTrialConfig(
    name="composed",
    ego_blueprint="vehicle.lincoln.mkz_2020",
    target_speed_kmh=TARGET_SPEED_KMH,
    speed_multiplier=1.0,
    use_vehicle_camera_defaults=True,
    controller_mode="behavior_agent",
    controller_behavior="normal",
)

runner = TrialRunner(verbose=False)

# Quick sanity-check by sampling the lambdas at a few risk levels.
print("Speed-function preview at v = {:.1f} km/h:".format(TARGET_SPEED_KMH))
for r_sample in (None, 0.0, 0.3, 0.6, 0.9):
    e2e_v      = e2e_speed_fn(TARGET_SPEED_KMH, r_sample)
    composed_v = composed_speed_fn(TARGET_SPEED_KMH, r_sample)
    r_str = "None " if r_sample is None else f"{r_sample:4.2f}"
    print(f"  r={r_str}  ->  e2e v'={e2e_v:5.2f} km/h   composed v'={composed_v:5.2f} km/h")
print(f"SAVE_FRAMES = {SAVE_FRAMES}")

## 4 — Load Inference Models
Loads `QueuedE2ERiskInference` and `QueuedComposedBDUGRURiskInference` once and reuses them across every trial. Both classes' `add_image_path(path)` is the per-frame streaming entry point we'll wrap below.

Whichever predictor's checkpoint is missing is set to `None` and that model's run is skipped for every trial.

In [ ]:
device_name = "cuda" if torch.cuda.is_available() else "cpu"

E2E_CHECKPOINT      = Path(Config.PATH_TO_MODELS) / "e2e_risk_checkpoint.pt"
COMPOSED_CHECKPOINT = Path(Config.PATH_TO_MODELS) / "bdu_gru_search_02.pt"
YOLO_CHECKPOINT     = Path(Config.PATH_TO_MODELS) / "yolo11s.pt"
DEPTH_CHECKPOINT    = Path(Config.PATH_TO_MODELS) / "depth_anything_v2_vits.pth"
CLIMATE_CHECKPOINT  = Path(Config.PATH_TO_MODELS) / "environment_multitask_checkpoint.pt"
ROAD_CHECKPOINT     = Path(Config.PATH_TO_MODELS) / "road_segmentation_multitask_checkpoint.pt"

print(f"Device: {device_name}")

# ---- E2E predictor ----------------------------------------------------------
e2e_predictor = None
if E2E_CHECKPOINT.is_file():
    e2e_predictor = QueuedE2ERiskInference.from_checkpoint(
        checkpoint_path=str(E2E_CHECKPOINT),
        device=device_name,
    )
    print(f"E2E loaded: {E2E_CHECKPOINT.name}")
else:
    print(f"[skip] E2E checkpoint missing: {E2E_CHECKPOINT}")

# ---- Composed BDU-GRU predictor --------------------------------------------
composed_predictor = None
required_for_composed = {
    "composed BDU-GRU": COMPOSED_CHECKPOINT,
    "YOLO":             YOLO_CHECKPOINT,
    "depth":            DEPTH_CHECKPOINT,
}
missing_required = [name for name, p in required_for_composed.items() if not p.is_file()]
if missing_required:
    print(f"[skip] Composed predictor missing: {', '.join(missing_required)}")
else:
    yolo_model = YOLO(str(YOLO_CHECKPOINT))
    depth_estimator = DepthAnythingV2Estimator(
        checkpoint_path=DEPTH_CHECKPOINT,
        encoder="vits",
        device=device_name,
    )
    environment_predictor = (
        create_environment_classifier_predictor(checkpoint_path=str(CLIMATE_CHECKPOINT), device=device_name)
        if CLIMATE_CHECKPOINT.is_file() else None
    )
    road_segmentation = (
        load_road_segmentation_model(checkpoint_path=str(ROAD_CHECKPOINT), device=device_name)
        if ROAD_CHECKPOINT.is_file() else None
    )
    composed_predictor = QueuedComposedBDUGRURiskInference.from_checkpoint(
        checkpoint_path=str(COMPOSED_CHECKPOINT),
        feature_integrator=FeatureIntegrator(),
        yolo_model=yolo_model,
        depth_estimator=depth_estimator,
        environment_predictor=environment_predictor,
        road_segmentation=road_segmentation,
        device=device_name,
    )
    print(f"Composed loaded: {COMPOSED_CHECKPOINT.name}")
    print(f"  environment_predictor: {environment_predictor is not None}")
    print(f"  road_segmentation:     {road_segmentation is not None}")

if e2e_predictor is None and composed_predictor is None:
    raise RuntimeError("Both predictors are unavailable - cannot run.")

### Per-Frame Predict Wrapper + Post-Run Cleanup
`make_predict_step_fn(predictor)` resets the queued predictor's temporal queue and returns a `Callable[[rgb_path], float | None]` suitable for `run_subtrial(predict_step_fn=...)`. The runner calls it every captured frame and stashes the result so the risk-aware speed modifier (which fires every tick) reads the most recent prediction.

`rename_predicted_risk(jsonl_path, new_field)` renames the JSONL's `predicted_risk` column to a model-specific name so the field is unambiguous downstream. Companion fields (`predicted_risk_window`, etc.) are dropped — only the scalar prediction survives the rename.

`cleanup_images(run_path)` deletes the run's `images/` directory; called when `SAVE_FRAMES=False`.

In [ ]:
def make_predict_step_fn(queued_predictor) -> Callable[[str], float | None]:
    """Reset the predictor's queue and return a per-frame inference closure."""
    queued_predictor.reset_queue()

    def _predict(rgb_path: str) -> float | None:
        out = queued_predictor.add_image_path(rgb_path)
        if out.ready and out.latest_risk is not None:
            return float(out.latest_risk)
        return None

    return _predict


def rename_predicted_risk(jsonl_path: Path, new_field: str) -> int:
    """Rename `predicted_risk` -> `new_field` per record. Returns count of records renamed."""
    if not jsonl_path.is_file():
        return 0

    records = load_jsonl_records(str(jsonl_path))
    if not records:
        return 0

    drop_fields = (
        "predicted_risk_window",
        "predicted_risk_ready",
        "predicted_risk_buffer_size",
        "predicted_risk_error",
    )
    renamed = 0
    for rec in records:
        if "predicted_risk" in rec:
            rec[new_field] = rec.pop("predicted_risk")
            renamed += 1
        for k in drop_fields:
            rec.pop(k, None)

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return renamed


def cleanup_images(run_path: Path) -> int:
    """Remove the run's images/ directory. Returns bytes freed (best-effort)."""
    images_dir = run_path / "images"
    if not images_dir.is_dir():
        return 0
    total = 0
    for p in images_dir.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass
    shutil.rmtree(images_dir, ignore_errors=True)
    return total


@dataclass
class _ModelRun:
    name: str
    field: str                                        # JSONL field name after rename
    ego_cfg: EgoTrialConfig
    predictor: object | None
    speed_fn: Callable[[float, float | None], float]


print("Helpers ready: make_predict_step_fn(), rename_predicted_risk(), cleanup_images()")

## 5 — Run Each Trial × Each Model
For every selected trial:
1. Build a fresh per-frame predict closure for the active model (resets the temporal queue per trial).
2. Run the subtrial with `predict_step_fn=...` and `risk_speed_fn=...` so the model's risk feeds the v' lambda live.
3. Rename `predicted_risk` → `predicted_risk_<model>` in the JSONL.
4. Optionally wipe the run's `images/` directory.

A predictor that didn't load is dropped from the schedule entirely. Per-(trial, model) failures are recorded and the batch keeps going.

In [ ]:
all_summaries: list = []
inference_stats: dict[str, dict] = {}
failed: list[tuple[str, str, str]] = []

# Build the active model_runs from whichever predictors loaded.
model_runs: list[_ModelRun] = []
if e2e_predictor is not None:
    model_runs.append(_ModelRun(name="e2e", field="predicted_risk_e2e",
                                ego_cfg=e2e_ego_cfg, predictor=e2e_predictor,
                                speed_fn=e2e_speed_fn))
if composed_predictor is not None:
    model_runs.append(_ModelRun(name="composed", field="predicted_risk_composed",
                                ego_cfg=composed_ego_cfg, predictor=composed_predictor,
                                speed_fn=composed_speed_fn))

if not model_runs:
    raise RuntimeError("No predictors loaded - cannot run.")

# --- Run-time accounting -----------------------------------------------------
fixed_delta_s    = float(Config.SIM_FIXED_DELTA_SECONDS)
stride_ticks     = int(Config.RECORD_EVERY_N_TICKS)
sim_time_cap_s   = MAX_STEPS_PER_TRIAL * fixed_delta_s
max_jsonl_frames = MAX_STEPS_PER_TRIAL // stride_ticks
record_hz        = 1.0 / (fixed_delta_s * stride_ticks)
total_runs = len(selected_trial_names) * len(model_runs)

print("Run configuration:")
print(f"  fixed_delta:          {fixed_delta_s:.3f} s/tick  ({1/fixed_delta_s:.1f} Hz physics)")
print(f"  image_stride:         {stride_ticks} ticks/frame  ({record_hz:.2f} Hz JSONL records)")
print(f"  max_steps_per_trial:  {MAX_STEPS_PER_TRIAL}  (cap = {sim_time_cap_s:.1f} s sim, <= {max_jsonl_frames} JSONL frames)")
print(f"  models active:        {[m.name for m in model_runs]}")
print(f"  trials selected:      {len(selected_trial_names)}  ->  {total_runs} subtrial runs total")
print(f"  SAVE_FRAMES:          {SAVE_FRAMES}  ({'keep RGB+topdown' if SAVE_FRAMES else 'delete images/ after run'})")

_progress_state: dict = {"trial_started": 0.0}

def _format_secs(s: float) -> str:
    if s < 60:
        return f"{s:5.1f}s"
    m, sec = divmod(s, 60)
    return f"{int(m):02d}:{int(sec):02d}"

def _on_progress(step: int, max_steps: int) -> None:
    now = time.time()
    last = _progress_state.get("last_print", 0.0)
    if now - last < 0.2:
        return
    _progress_state["last_print"] = now
    elapsed = now - _progress_state["trial_started"]
    rate = step / elapsed if elapsed > 0 else 0.0
    pct = 100.0 * step / max_steps if max_steps else 0.0
    eta_to_cap = (max_steps - step) / rate if rate > 0 else float("inf")
    speedup = (step * fixed_delta_s) / elapsed if elapsed > 0 else 0.0
    sys.stdout.write(
        f"\r      step {step:5d}/{max_steps} ({pct:5.1f}% of cap)  "
        f"wall={_format_secs(elapsed)}  "
        f"rate={rate:5.1f} t/s  "
        f"speedup={speedup:4.2f}x  "
        f"eta_to_cap={_format_secs(eta_to_cap) if rate > 0 else '  inf'}    "
    )
    sys.stdout.flush()

batch_started = time.time()
run_idx = 0
for t_idx, trial_name in enumerate(selected_trial_names, start=1):
    print()
    print(f">>> [{t_idx}/{len(selected_trial_names)}] {trial_name}", flush=True)
    try:
        trial = TrialDefinition.load(trial_name)
    except Exception as e:
        for mr in model_runs:
            failed.append((trial_name, mr.name, f"load: {type(e).__name__}: {e}"))
        print(f"    [skip] failed to load trial: {type(e).__name__}: {e}")
        continue

    for mr in model_runs:
        run_idx += 1
        print(f"  -- model={mr.name}  ({run_idx}/{total_runs})", flush=True)
        _progress_state["trial_started"] = time.time()
        _progress_state["last_print"] = 0.0
        try:
            # Fresh per-trial predict closure (resets the temporal queue) so each
            # run starts from a clean burn-in instead of inheriting the previous trial.
            predict_fn = make_predict_step_fn(mr.predictor)

            summary = runner.run_subtrial(
                trial=trial,
                ego_cfg=mr.ego_cfg,
                max_steps=MAX_STEPS_PER_TRIAL,
                image_stride=Config.RECORD_EVERY_N_TICKS,
                store_rgb_images=True,                       # required by the streaming predictor
                store_topdown_images=SAVE_FRAMES,
                topdown_capture_delay_ticks=TOPDOWN_CAPTURE_DELAY,
                topdown_resolution=TOPDOWN_RESOLUTION,
                topdown_fov=TOPDOWN_FOV,
                topdown_align_risk_rotation=TOPDOWN_ALIGN_RISK,
                store_risk_frame_images=False,
                store_static_risk_map=False,
                draw_debug_every_tick=SAVE_FRAMES,           # only paint debug if frames are kept
                predict_step_fn=predict_fn,
                risk_speed_fn=mr.speed_fn,
                progress_callback=_on_progress,
                progress_every_n_steps=25,
            )
            elapsed = time.time() - _progress_state["trial_started"]
            sim_seconds = summary.num_frames * fixed_delta_s * stride_ticks
            speedup = sim_seconds / elapsed if elapsed > 0 else float("inf")
            sys.stdout.write("\r" + " " * 130 + "\r")
            sys.stdout.flush()

            jsonl_path = Path(summary.run_path) / "dataset.jsonl"
            renamed = rename_predicted_risk(jsonl_path, mr.field)
            inference_stats[f"{trial_name}|{mr.name}"] = {"frames": summary.num_frames, "predicted": renamed}

            print(
                f"      ok in {elapsed:6.1f}s wall  "
                f"frames={summary.num_frames:4d}  sim_t={sim_seconds:6.1f}s  "
                f"speedup={speedup:5.2f}x  dist={summary.traveled_m:7.1f}m  "
                f"{mr.field}={renamed}  finished={summary.finished}"
            )

            if not SAVE_FRAMES:
                freed = cleanup_images(Path(summary.run_path))
                print(f"      images/ wiped ({freed/1e6:.1f} MB freed)")

            all_summaries.append((trial_name, mr.name, summary))
        except Exception as e:
            elapsed = time.time() - _progress_state["trial_started"]
            sys.stdout.write("\r" + " " * 130 + "\r")
            sys.stdout.flush()
            print(f"      FAILED in {elapsed:.1f}s: {type(e).__name__}: {e}")
            failed.append((trial_name, mr.name, f"{type(e).__name__}: {e}"))

batch_elapsed = time.time() - batch_started
print()
print(f"Batch done in {batch_elapsed:.1f}s (ok={len(all_summaries)}, failed={len(failed)})")

## 6 — Summary

In [ ]:
if not all_summaries and not failed:
    print("No trials were run.")
else:
    print(f"=== Successful runs ({len(all_summaries)}) ===")
    print(
        f"  {'trial_name':50s} {'model':>10s} {'frames':>6s} {'dist_m':>9s} "
        f"{'risk_auc':>9s} {'risk/m':>9s} {'preds':>6s}  finished  run_path"
    )
    for trial_name, model_name, s in all_summaries:
        stats = inference_stats.get(f"{trial_name}|{model_name}", {})
        preds = stats.get("predicted", 0)
        print(
            f"  {trial_name:50s} {model_name:>10s} {s.num_frames:6d} {s.traveled_m:9.1f} "
            f"{s.risk_auc:9.3f} {s.risk_per_meter:9.5f} {preds:6d}  "
            f"{str(s.finished):>8s}  {s.run_path}"
        )

    if failed:
        print()
        print(f"=== Failed runs ({len(failed)}) ===")
        for trial_name, model_name, err in failed:
            print(f"  {trial_name} / {model_name}: {err}")